# 0.1 一次生成的全过程 / One generation, start to finish

文本进去，下一个 token 出来。把中间的六步走一遍，认识 token、logits、贪心解码和采样。

Text goes in, the next token comes out. Walk through the six steps in between and meet tokens, logits, greedy decoding and sampling.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手动走完一次前向计算 / Walk through one forward pass by hand

不直接调用 `model(input_ids)`，而是用 GPT-2 的各个部件依次完成嵌入、12 层 block、最后的归一化和 unembedding，得到 logits。然后取最后一个位置，算出概率最高的 5 个 token。测试会把你的 logits 与模型自己的输出逐项比较。

Without calling `model(input_ids)`, use the parts of GPT-2 to do the embedding, the 12 blocks, the final normalization and the unembedding in turn and obtain the logits. Then take the last position and compute the 5 most probable tokens. The tests compare your logits with the model's own output entry by entry.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# The parts of GPT-2 small you need:
#   t = model.transformer
#   t.wte   token embedding table,     shape (50257, 768)
#   t.wpe   position embedding table,  shape (1024, 768)
#   t.h     a list of 12 Transformer blocks
#   t.ln_f  the final LayerNorm
#   model.lm_head   the "unembedding": 768 numbers -> 50257 scores (logits)


def run_block(block, x):
    """Call one Transformer block. Works across transformers versions."""
    out = block(x)
    return out[0] if isinstance(out, tuple) else out


@torch.no_grad()
def manual_logits(model, input_ids):
    """Rebuild model(input_ids).logits step by step.

    input_ids: long tensor (1, T).  Returns float tensor (1, T, 50257).
    """
    t = model.transformer
    # TODO 1: look up the token embeddings and the position embeddings
    #         (positions are 0, 1, ..., T-1) and add them           -> (1, T, 768)
    # TODO 2: pass the result through the 12 blocks, one after another
    # TODO 3: apply the final LayerNorm, then the unembedding        -> (1, T, 50257)
    raise NotImplementedError


@torch.no_grad()
def next_token_top_k(model, tokenizer, text, k=5):
    """The k most likely next tokens after `text`.

    Returns a list of (token_string, probability) pairs, most likely first.
    """
    # TODO 4: tokenise, get the logits, keep only the LAST position
    # TODO 5: softmax -> probabilities, take the top k, decode each token id
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()

text = "The capital of France is"
ids = tok(text, return_tensors="pt").input_ids
assert ids.tolist() == [[464, 3139, 286, 4881, 318]], "five tokens: The / capital / of / France / is"

with torch.no_grad():
    ref = gpt2(ids).logits
out = manual_logits(gpt2, ids)
assert out.shape == (1, 5, 50257), f"one row of 50257 scores per input token, got {tuple(out.shape)}"
assert torch.allclose(out, ref, atol=1e-3), "your step-by-step logits differ from model(input_ids).logits"

top = next_token_top_k(gpt2, tok, text, k=5)
assert len(top) == 5 and all(isinstance(s, str) and isinstance(p, float) for s, p in top)
probs = [p for _, p in top]
assert probs == sorted(probs, reverse=True), "most likely first"
ref_p = torch.softmax(ref[0, -1], -1)
assert top[0][0] == tok.decode(ref_p.argmax()) and abs(top[0][1] - ref_p.max().item()) < 1e-4, "use the logits of the last position"
assert 0 < sum(probs) < 1, "five tokens out of 50257 cannot hold all the probability"
print("m01_forward: all tests passed. Top 5 after", repr(text))
for s, p in top:
    print(f"  {s!r:12} {p:.3f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

位置编号用 `torch.arange(T)` 生成，`t.wte(input_ids)` 和 `t.wpe(positions)` 的结果直接相加，形状会自动广播。

Create the positions with `torch.arange(T)`. Add `t.wte(input_ids)` and `t.wpe(positions)` directly and broadcasting takes care of the shapes.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

循环里写 `x = run_block(block, x)`。12 层结束后是 `model.lm_head(t.ln_f(x))`。

Inside the loop write `x = run_block(block, x)`. After the 12 layers comes `model.lm_head(t.ln_f(x))`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

logits 的形状是 `(1, T, 50257)`。预测下一个 token 只用最后一行，即 `logits[0, -1]`。

The logits have shape `(1, T, 50257)`. Only the last row predicts the next token, `logits[0, -1]`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def run_block(block, x):
    out = block(x)
    return out[0] if isinstance(out, tuple) else out


@torch.no_grad()
def manual_logits(model, input_ids):
    t = model.transformer
    T = input_ids.shape[1]
    x = t.wte(input_ids) + t.wpe(torch.arange(T))     # one 768-d vector per token
    for block in t.h:                                 # 12 rounds of attention + MLP
        x = run_block(block, x)
    return model.lm_head(t.ln_f(x))                   # one score per vocabulary entry


@torch.no_grad()
def next_token_top_k(model, tokenizer, text, k=5):
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    last = manual_logits(model, input_ids)[0, -1]     # only the last position predicts the next token
    probs = torch.softmax(last, dim=-1)
    top = probs.topk(k)
    return [(tokenizer.decode(i), p.item()) for p, i in zip(top.values, top.indices)]
```

</details>

## 2. 手写贪心生成循环 / A greedy generation loop by hand

实现贪心解码，每一步取 logit 最大的 token 并接到输入后面。测试会与 `model.generate(do_sample=False)` 的结果对比。再写一个统计重复 n-gram 的小函数，用它确认贪心输出确实陷入了重复。运行后请看一眼打印出来的文本。

Implement greedy decoding by taking the token with the largest logit at each step and appending it to the input. The tests compare with `model.generate(do_sample=False)`. Then write a small function that counts repeated n-grams and use it to confirm that the greedy output does fall into repetition. Read the printed text after running.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


@torch.no_grad()
def greedy_generate(model, input_ids, max_new_tokens=20):
    """Greedy decoding: at every step append the single most likely token.

    input_ids: long tensor (1, T).  Returns long tensor (1, T + max_new_tokens).
    Do not call model.generate.
    """
    for _ in range(max_new_tokens):
        # TODO 1: logits = model(input_ids).logits, keep the last position   -> (50257,)
        # TODO 2: next_id = the index of the largest logit
        # TODO 3: append next_id to input_ids (torch.cat along dim 1)
        raise NotImplementedError
    return input_ids


def repeated_ngrams(token_ids, n=4):
    """How many n-grams in the list `token_ids` have already appeared earlier in it."""
    # TODO 4: walk over all windows of length n, count the ones seen before
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()

assert repeated_ngrams([1, 2, 3, 4, 5, 6], n=4) == 0
assert repeated_ngrams([1, 2, 3, 4, 1, 2, 3, 4, 1], n=4) == 2, "(1,2,3,4) and (2,3,4,1) each appear a second time"

ids = tok("The meaning of life is", return_tensors="pt").input_ids
out = greedy_generate(gpt2, ids, max_new_tokens=40)
assert out.shape == (1, ids.shape[1] + 40)
assert torch.equal(out[:, :ids.shape[1]], ids), "the prompt must stay at the front"
ref = gpt2.generate(ids, do_sample=False, max_new_tokens=40, pad_token_id=tok.eos_token_id)
assert torch.equal(out, ref), "differs from model.generate(do_sample=False)"

# Greedy decoding is deterministic.
assert torch.equal(out, greedy_generate(gpt2, ids, max_new_tokens=40))

# ... and it gets stuck in a loop. This is the reason sampling exists.
reps = repeated_ngrams(out[0].tolist(), n=4)
assert reps >= 10, "greedy GPT-2 output is expected to repeat itself here"
print("m01_greedy: all tests passed")
print(f"  {reps} repeated 4-grams in:", repr(tok.decode(out[0])))

<details><summary><b>提示 1 / Hint 1</b></summary>

取最大值不需要先做 softmax，因为 softmax 不改变大小顺序，直接对 logits 用 `argmax` 即可。

No softmax is needed before taking the maximum, because softmax keeps the order. Use `argmax` on the logits directly.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`next_id` 是 0 维张量，用 `next_id.view(1, 1)` 变成 `(1, 1)` 后再 `torch.cat([input_ids, ...], dim=1)`。

`next_id` is a 0-dim tensor. Reshape it with `next_id.view(1, 1)` and then `torch.cat([input_ids, ...], dim=1)`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

统计重复时用一个 `set` 记录见过的 n-gram，把每个窗口转成 `tuple` 才能放进集合。

For the repeat count keep a `set` of n-grams already seen. Turn each window into a `tuple` so it can go into the set.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


@torch.no_grad()
def greedy_generate(model, input_ids, max_new_tokens=20):
    for _ in range(max_new_tokens):
        logits = model(input_ids).logits[0, -1]            # scores for the next token
        next_id = logits.argmax()                          # softmax is not needed: it keeps the order
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
    return input_ids


def repeated_ngrams(token_ids, n=4):
    seen, repeats = set(), 0
    for i in range(len(token_ids) - n + 1):
        gram = tuple(token_ids[i:i + n])
        repeats += gram in seen
        seen.add(gram)
    return repeats
```

</details>

## 3. 手写带温度的采样 / Sampling with temperature by hand

实现从 `softmax(logits / T)` 中抽取一个 token，并把它接进生成循环。测试会检查四件事，即大量抽样后的频率接近概率、温度很低时等同于贪心、同一个随机种子给出同样的输出、采样文本的重复少于贪心文本。

Implement drawing one token from `softmax(logits / T)` and plug it into the generation loop. The tests check four things, namely that frequencies over many draws approach the probabilities, that a very low temperature equals greedy, that the same random seed gives the same output, and that sampled text repeats less than greedy text.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def sample_next(logits, temperature=1.0, generator=None):
    """Draw one token id from softmax(logits / temperature).

    logits: float tensor (V,).  Returns a 0-dim long tensor.
    """
    # TODO 1: divide the logits by the temperature, then softmax
    # TODO 2: draw one index with torch.multinomial(probs, 1, generator=generator)
    raise NotImplementedError


@torch.no_grad()
def sample_generate(model, input_ids, max_new_tokens=20, temperature=1.0, seed=0):
    """Like greedy_generate, but each new token is sampled. Same seed, same output."""
    g = torch.Generator().manual_seed(seed)
    for _ in range(max_new_tokens):
        # TODO 3: logits of the last position -> sample_next(..., generator=g) -> append
        raise NotImplementedError
    return input_ids

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Sampling follows the probabilities: draw many times and count.
z = torch.tensor([2.0, 1.0, 0.1])
g = torch.Generator().manual_seed(0)
draws = torch.stack([sample_next(z, 1.0, g) for _ in range(20000)])
freq = torch.bincount(draws, minlength=3) / 20000
assert torch.allclose(freq, torch.softmax(z, -1), atol=0.015), f"frequencies {freq.tolist()} should be close to softmax(z) = (0.659, 0.242, 0.099)"

# 2. Temperature reshapes the distribution.
g = torch.Generator().manual_seed(0)
cold = torch.stack([sample_next(z, 0.05, g) for _ in range(500)])
assert (cold == 0).all(), "at a very low temperature only the top token is drawn"
g = torch.Generator().manual_seed(0)
hot = torch.stack([sample_next(z, 100.0, g) for _ in range(20000)])
assert torch.allclose(torch.bincount(hot, minlength=3) / 20000, torch.full((3,), 1 / 3), atol=0.015), "at a very high temperature all tokens are about equally likely"

# 3. On the real model.
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()
ids = tok("The meaning of life is", return_tensors="pt").input_ids

a = sample_generate(gpt2, ids, 40, temperature=1.0, seed=1)
b = sample_generate(gpt2, ids, 40, temperature=1.0, seed=1)
c = sample_generate(gpt2, ids, 40, temperature=1.0, seed=2)
assert a.shape == (1, ids.shape[1] + 40)
assert torch.equal(a, b), "same seed, same output"
assert not torch.equal(a, c), "a different seed should give a different continuation"

greedy = gpt2.generate(ids, do_sample=False, max_new_tokens=40, pad_token_id=tok.eos_token_id)
assert torch.equal(sample_generate(gpt2, ids, 40, temperature=1e-4, seed=3), greedy), "temperature near 0 should reproduce greedy decoding"

def _repeats(x, n=4):
    seen, r = set(), 0
    for i in range(len(x) - n + 1):
        gram = tuple(x[i:i + n]); r += gram in seen; seen.add(gram)
    return r
assert _repeats(a[0].tolist()) < _repeats(greedy[0].tolist()), "sampled text should repeat less than greedy text"
print("m01_sample: all tests passed")
print("  greedy :", repr(tok.decode(greedy[0])))
print("  seed 1 :", repr(tok.decode(a[0])))
print("  seed 2 :", repr(tok.decode(c[0])))

<details><summary><b>提示 1 / Hint 1</b></summary>

`torch.multinomial(probs, 1, generator=generator)` 按给定概率抽一个下标，返回形状为 `(1,)` 的张量，取 `[0]` 得到 0 维张量。

`torch.multinomial(probs, 1, generator=generator)` draws one index with the given probabilities and returns a tensor of shape `(1,)`. Take `[0]` for a 0-dim tensor.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

生成循环与上一题相同，只是把 `argmax` 换成 `sample_next(logits, temperature, generator=g)`。随机数生成器 `g` 要在循环外面创建一次。

The loop is the same as in the previous exercise with `argmax` replaced by `sample_next(logits, temperature, generator=g)`. Create the generator `g` once, outside the loop.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def sample_next(logits, temperature=1.0, generator=None):
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)[0]


@torch.no_grad()
def sample_generate(model, input_ids, max_new_tokens=20, temperature=1.0, seed=0):
    g = torch.Generator().manual_seed(seed)
    for _ in range(max_new_tokens):
        logits = model(input_ids).logits[0, -1]
        next_id = sample_next(logits, temperature, generator=g)
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
    return input_ids
```

</details>